# Recommandation d'artistes avec XGBoost

Toute la logique se trouve dans `src/recommend_xgboost.py`. Ce notebook charge une version précise des données, sélectionne la meilleure configuration sur la validation, entraîne le modèle final et montre son utilisation.

In [ ]:
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")

import importlib
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Impossible de trouver la racine MusicRecoSystem")
sys.path.insert(0, str(PROJECT_ROOT))

import src.recommend_xgboost as xgb_module
xgb_module = importlib.reload(xgb_module)

## 1. Charger les données nettoyées

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "listens.parquet"
listens = pd.read_parquet(DATA_PATH)

pd.Series({
    "dataset_version": listens["dataset_version"].iloc[0],
    "interactions": len(listens),
    "users": listens["user"].nunique(),
    "artist_identities": listens["artist_uid"].nunique(),
})

## 2. Sélectionner la configuration et entraîner le modèle final

Les configurations sont comparées avec le NDCG@10 de validation. Le meilleur modèle est ensuite réentraîné sur train + validation et évalué sur le test.

In [ ]:
import src.recommend_xgboost as xgb_module
xgb_module = importlib.reload(xgb_module)

experiment = xgb_module.select_and_fit_best_xgboost(
    listens,
    top_n=10,
    random_state=42,
    verbose=True,
)

display(experiment.tuning_results)
display(pd.DataFrame({
    "validation": experiment.validation_metrics,
    "test_final": experiment.test_metrics,
}))
print(f"Dataset : {experiment.dataset_version}")
print(f"Meilleure configuration : {experiment.best_config}")

## 3. Comprendre les variables du modèle

In [ ]:
experiment.recommender.feature_importance().to_frame()

## 4. Obtenir 10 recommandations

In [ ]:
USER = "BKILPATRICK11"

experiment.recommender.recommend(USER, top_n=10)